# Fine-tuning LoRA do Qwen2.5-0.5B — sentimento de reviews PT-BR

Ative **GPU T4** e **Internet** em *Settings*. Adicione seu token do Hugging Face (com escrita) em *Add-ons → Secrets* com o nome `HF_TOKEN`. Rode com *Save Version → Save & Run All*.

In [ ]:
%cd /kaggle/working
!rm -rf reviews-ptbr-llm
!git clone https://github.com/rmascheti/reviews-ptbr-llm.git
%cd reviews-ptbr-llm
!pip install -q peft
# o Kaggle traz torchao 0.10, e o peft recusa versões abaixo de 0.16 mesmo sem usar quantização
!pip uninstall -y -q torchao
!python src/prepare_data.py

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
# registra o ambiente usado no treino
!git log --oneline -1
import torch, transformers, peft
from huggingface_hub import whoami
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA")
print("transformers", transformers.__version__, "| peft", peft.__version__)
print("HF:", whoami()["name"])

In [ ]:
!python src/train_lora.py --train-size 40000 --push-to-hub rmascheti/qwen2.5-0.5b-reviews-ptbr

In [ ]:
!python src/baseline.py
!cat reports/baseline.json reports/lora.json